# PyTorch Experience · 02
## `model(x)` vs `model.forward(x)` · Forward Hooks

**One glance:** Both calls can perform the same forward computation, but **`model(x)` also runs PyTorch's module-call machinery**, including hooks on that module.

**Official docs:** [nn.Module](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html) · [register_forward_hook()](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.register_forward_hook)

[← Pytorch Experience](https://lawson-dong.vercel.app/coding/pytorch-experience)

### 1 · The difference

![Diagram comparing model(x) with model.forward(x)](https://raw.githubusercontent.com/Lawson-Dong/personal-website/main/public/images/pytorch-model-call-vs-forward.svg)

| Call | Behavior |
|---|---|
| `model(x)` | Calls `forward()` through `nn.Module`; registered hooks on the module can run |
| `model.forward(x)` | Calls `forward()` directly; skips hooks registered on that module |

**Use `model(x)` in normal code.** Hooks on child modules may still run if their modules are called inside `forward()`.

### 2 · Same calculation, different call paths

Small CPU-friendly example — run it in Colab.

In [ ]:
import torch
from torch import nn

class Double(nn.Module):
    def forward(self, x):
        return x * 2

model = Double()
x = torch.tensor([1., 2.])

print("model(x):        ", model(x))
print("model.forward(x):", model.forward(x))

### 3 · What is a Forward Hook?

![Without hooks versus with hooks: saving intermediate features](https://raw.githubusercontent.com/Lawson-Dong/personal-website/main/public/images/pytorch-hook-purpose.svg)

A **Forward Hook** runs after a chosen module's forward computation. It can **observe or save that module's output without editing `forward()`**.

Signature: `hook(module, inputs, output)`. A hook that only reads or saves outputs does not change the computed result.

In [ ]:
def show_hook(module, inputs, output):
    print("HOOK ran! Output:", output)

handle = model.register_forward_hook(show_hook)

print("1) model(x)")
model(x)           # Hook runs

print("2) model.forward(x)")
model.forward(x)   # Hook on model is skipped

handle.remove()    # Remove the hook when finished

### 4 · Save an intermediate layer output

In a two-layer model, the usual return value is the **final output**. A hook lets us save `layer1` as well.

In [ ]:
class TwoLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(3, 4)
        self.layer2 = nn.Linear(4, 2)

    def forward(self, x):
        x = self.layer1(x)
        return self.layer2(x)

network = TwoLayer()
features = {}

def save_layer1(module, inputs, output):
    features["layer1"] = output.detach()

handle = network.layer1.register_forward_hook(save_layer1)
final = network(torch.ones(1, 3))

print("Final output:", final.shape)          # [1, 2]
print("Layer 1 output:", features["layer1"].shape)  # [1, 4]

handle.remove()

### Remember

- **`model(x)`** → recommended; forwards through PyTorch's module call mechanism.
- **`model.forward(x)`** → calls the method directly; skips hooks on that module.
- **Forward Hook** → inspect or save intermediate outputs without changing the model's `forward()` implementation.
- **`handle.remove()`** → unregister the hook.
- Hooks can optionally **replace** outputs if written to return a replacement; our examples only observe.

**Official references:** [PyTorch nn.Module](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html) · [register_forward_hook()](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.register_forward_hook) · [register_forward_pre_hook()](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.register_forward_pre_hook)